# Decision Tree Model Implementation
This notebook implements a Decision Tree model for the dataset. It includes data loading, model training, evaluation, and visualization of the decision tree.

In [2]:
# Import necessary libraries
import os
import pandas as pd
import geopandas as gpd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
from sklearn import tree


Pfad muss ich noch so anpassen, dass der Pfad nur noch \datei beinhaltet und somit Lars auch drauf zugreifen kann

In [3]:

# Load the dataset
data_path = os.getenv(rf'C:\Users\elbma\Nextcloud2\Shared\AI_for_HWS_final_project\data')
data_file = r'C:\\Users\\elbma\\Nextcloud2\\Shared\AI_for_HWS_final_project\data\\green_roofs_with_floors_cleaned.csv'
data = pd.read_csv(data_file)


In [4]:
# Preprocess the data
print(data.columns)  # Ausgabe der Spaltennamen
print(data.head())   # Erste 5 Zeilen anschauen
print(data.shape)    # Dimensionen

Index(['gml_id_lef', 'importid', 'geb_nutz', 'gruendach', 'ex_int', 'gex20_p',
       'geb_area', 'nutz', 'ext', 'egeb_nutz', 'egruendach', 'eex_int',
       'index_righ', 'aog', 'aug'],
      dtype='str')
     gml_id_lef  importid    geb_nutz        gruendach    ex_int  gex20_p  \
0  d_gebaeude.1         1  Tiefgarage        vorhanden  intensiv     0.90   
1  d_gebaeude.2         2  Tiefgarage        vorhanden  intensiv     8.73   
2  d_gebaeude.3         3  Tiefgarage  nicht vorhanden       NaN     0.00   
3  d_gebaeude.4         4  Tiefgarage        vorhanden  intensiv    23.30   
4  d_gebaeude.5         5  Tiefgarage  nicht vorhanden       NaN     0.00   

   geb_area                nutz  ext            egeb_nutz    egruendach  \
0    244.19  Tiefgarage (ALKIS)    0  Underground parking      existent   
1    908.26  Tiefgarage (ALKIS)    0  Underground parking      existent   
2    435.39  Tiefgarage (ALKIS)    0  Underground parking  non-existent   
3    871.58  Tiefgarage (ALKIS)

In [5]:
# Cell 6: Split the data into training and testing sets
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

# Step 1: Target-Spalte kodieren
target_col = "gruendach"
X = data.drop(columns=[target_col]).copy()
y = data[target_col].copy()

# Target: Text → Zahlen (z.B. "vorhanden" → 1, "nicht vorhanden" → 0)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Classes:", le.classes_)
print(f"y original: {y.unique()}")
print(f"y encoded: {set(y_encoded)}")

# Step 2: Nur numerische Features verwenden
X_numeric = X.select_dtypes(include=["number"]).copy()
print(f"\nFeatures: {X_numeric.columns.tolist()}")

# Step 3: Fehlende Werte füllen
X_numeric = X_numeric.fillna(X_numeric.median(numeric_only=True))
print(f"NaN nach fillna: {X_numeric.isna().sum().sum()}")

# Step 4: Train/Test aufteilen (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X_numeric, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"\nTrain-Größe: {X_train.shape[0]}")
print(f"Test-Größe: {X_test.shape[0]}")
print(f"Features: {X_train.shape[1]}")

Classes: ['nicht vorhanden' 'vorhanden']
y original: <StringArray>
['vorhanden', 'nicht vorhanden']
Length: 2, dtype: str
y encoded: {np.int64(0), np.int64(1)}

Features: ['importid', 'gex20_p', 'geb_area', 'ext', 'index_righ', 'aog', 'aug']
NaN nach fillna: 0

Train-Größe: 503732
Test-Größe: 125934
Features: 7


In [6]:
# Cell 7: Trainiere Decision Tree
model = DecisionTreeClassifier(
    random_state=42,
    max_depth=8,
    min_samples_leaf=20
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# Evaluation e.g. Accuracy

In [7]:
# Cell 9: Evaluation
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9994679752886433
[[121845      0]
 [    67   4022]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    121845
           1       1.00      0.98      0.99      4089

    accuracy                           1.00    125934
   macro avg       1.00      0.99      1.00    125934
weighted avg       1.00      1.00      1.00    125934



In [8]:
# Fallback, falls Zellen in anderer Reihenfolge ausgeführt wurden
if "target_col" not in globals():
	target_col = "gruendach"

# Check 1: Target wirklich nicht in X?
print("target_col in X?", target_col in X_numeric.columns)

# Check 2: Klassenverteilung
print(pd.Series(y_encoded).value_counts(normalize=True))

# Check 3: Verdächtige Spalten
print(X_numeric.columns.tolist())

target_col in X? False
0    0.967529
1    0.032471
Name: proportion, dtype: float64
['importid', 'gex20_p', 'geb_area', 'ext', 'index_righ', 'aog', 'aug']


In [9]:
# IDs/Join-Spalten entfernen
leak_cols = ["importid", "index_righ", "ext"]
X_clean = X_numeric.drop(columns=leak_cols, errors="ignore")

X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

model = DecisionTreeClassifier(random_state=42, max_depth=8, min_samples_leaf=20)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9994679752886433
[[121845      0]
 [    67   4022]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    121845
           1       1.00      0.98      0.99      4089

    accuracy                           1.00    125934
   macro avg       1.00      0.99      1.00    125934
weighted avg       1.00      1.00      1.00    125934



In [10]:
import numpy as np

y_shuffled = np.random.permutation(y_encoded)
X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y_shuffled, test_size=0.2, random_state=42, stratify=y_shuffled
)

model = DecisionTreeClassifier(random_state=42, max_depth=8, min_samples_leaf=20)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Shuffled-Target Accuracy:", accuracy_score(y_test, y_pred))

Shuffled-Target Accuracy: 0.9675306112725713


# Baseline

In [15]:
# Cell 9: Baseline-Vergleich
baseline_pred = [pd.Series(y_test).mode()[0]] * len(y_test)
baseline_acc = accuracy_score(y_test, baseline_pred)

print(f"Baseline Accuracy: {baseline_acc:.2%}")
print(f"Decision Tree Accuracy: {accuracy_score(y_test, y_pred):.2%}")

Baseline Accuracy: 96.75%
Decision Tree Accuracy: 99.95%


In [ ]:


# Create and train the Decision Tree model
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Visualize the Decision Tree
plt.figure(figsize=(20,10))
#tree.plot_tree(model, filled=True)
plt.title('Decision Tree Visualization')
plt.show()